# NB36 — GLiM Lithology Integration: Solidifying PC1/PC2 Interpretation

Joins USGS MRDS GLiM (Global Lithological Map, 0.25°) to the 4,554 complete-case PCA samples from NB35. Tests whether PC2 scores (Hg/Se hydrothermal axis) are significantly elevated in volcanic lithology classes vs unconsolidated sediment, while PC1 scores (lithogenic crustal axis) are indifferent to lithology.

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from itertools import combinations
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, METAL_COLORS, FIGW, ROW_H, grid_h
apply_style()

ROOT  = Path('/home/hmacgregor/BERIL-research-observatory')
CME   = ROOT / 'projects/comprehensive_metal_ecology'
DATA  = CME / 'data'
FIGS  = CME / 'figures'
GLIM  = Path('/home/hmacgregor/data/envdbs/global_lithology_glim.parquet')

assert GLIM.exists(), f'Missing GLiM: {GLIM}'
print('Setup done.')


Setup done.


In [2]:
# ── Reproduce NB35 PCA (same metal selection + NA filter) ──────────────────
samp = pd.read_parquet(DATA / 'nb33_sample_master.parquet')

METAL_COLS = sorted([c for c in samp.columns
                     if c.startswith('usgs_')
                     and samp[c].dtype in [np.float64, np.float32, float]])

metals_log = np.log1p(samp[METAL_COLS].copy())
metals_log.columns = [c.replace('usgs_', '') for c in METAL_COLS]

na_frac = metals_log.isna().mean()
keep_cols = na_frac[na_frac < 0.5].index.tolist()
metals_log = metals_log[keep_cols]
metals_complete = metals_log.dropna()
print(f'Complete-case samples: {len(metals_complete)}, metals: {len(keep_cols)}')

scaler = StandardScaler()
X = scaler.fit_transform(metals_complete)

pca = PCA(n_components=min(10, X.shape[1]))
pca.fit(X)
loadings = pd.DataFrame(
    pca.components_.T,
    index=metals_complete.columns,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)],
)
pct = pca.explained_variance_ratio_ * 100

scores = pd.DataFrame(
    pca.transform(X),
    index=metals_complete.index,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)],
)
scores = scores.join(samp[['lat', 'lon']], how='left')
print(f'PC1={pct[0]:.1f}%, PC2={pct[1]:.1f}%')
print(f'Hg loading — PC1={loadings.loc["hg","PC1"]:.3f}, PC2={loadings.loc["hg","PC2"]:.3f}')


In [3]:
# ── Join GLiM to PCA sample scores ─────────────────────────────────────────
glim = pd.read_parquet(GLIM).dropna()

# Aggregate: one GLiM row per 0.25° cell (take mode if multiple polygons overlap)
glim['lat_r'] = (glim['lat'] / 0.25).round() * 0.25
glim['lon_r'] = (glim['lon'] / 0.25).round() * 0.25
glim_grid = (glim.groupby(['lat_r', 'lon_r'])['lithology_class']
             .agg(lambda x: x.mode()[0])
             .reset_index())

scores['lat_r'] = (scores['lat'] / 0.25).round() * 0.25
scores['lon_r'] = (scores['lon'] / 0.25).round() * 0.25

merged = scores.merge(glim_grid, on=['lat_r', 'lon_r'], how='left')
# Also join raw Hg for validation
merged['usgs_hg'] = samp.loc[merged.index, 'usgs_hg'].values

coverage = merged['lithology_class'].notna().mean()
print(f'GLiM coverage: {merged["lithology_class"].notna().sum()}/{len(merged)} ({coverage*100:.1f}%)')
print('\nLithology class breakdown:')
print(merged['lithology_class'].value_counts().to_string())

# Group volcanic vs sedimentary
VOLCANIC = ['Intermediate Volcanic (VI)', 'Acid Volcanic (VA)',
            'Pyroclastics (PY)', 'Basic Volcanic (VB)']
SEDIMENT = ['Unconsolidated Sediment (SU)', 'Siliciclastic Sedimentary (SS)',
            'Carbonate Sedimentary (SC)', 'Evaporites (EV)', 'Mixed Sedimentary (SM)']
PLUTONIC = ['Acid Plutonic (PA)', 'Basic Plutonic (PB)', 'Intermediate Plutonic (PI)',
            'Metamorphic (MT)']

def broad_class(s):
    if s in VOLCANIC: return 'Volcanic'
    if s in SEDIMENT: return 'Sedimentary'
    if s in PLUTONIC: return 'Plutonic/Metamorphic'
    return None

merged['broad_class'] = merged['lithology_class'].apply(
    lambda x: broad_class(x) if pd.notna(x) else None)

print('\nBroad class breakdown:')
print(merged['broad_class'].value_counts().to_string())


In [4]:
# ── Kruskal-Wallis: PC1 and PC2 vs broad lithology class ───────────────────
from scipy.stats import kruskal, mannwhitneyu

lithology_ok = merged.dropna(subset=['broad_class'])
classes = lithology_ok['broad_class'].unique()

for pc in ['PC1', 'PC2']:
    groups = [lithology_ok.loc[lithology_ok['broad_class'] == c, pc].values
              for c in classes]
    H, p = kruskal(*groups)
    print(f'\nKruskal-Wallis {pc} ~ broad_class:  H={H:.2f}, p={p:.3e}')
    for c in sorted(classes):
        grp = lithology_ok.loc[lithology_ok['broad_class'] == c, pc]
        print(f'  {c:<25} median={grp.median():.3f}, n={len(grp)}')

print()

# Pairwise MWU: Volcanic vs Sedimentary for PC1 and PC2
for pc in ['PC1', 'PC2']:
    vol  = lithology_ok.loc[lithology_ok['broad_class'] == 'Volcanic',     pc].values
    sed  = lithology_ok.loc[lithology_ok['broad_class'] == 'Sedimentary',  pc].values
    U, p = mannwhitneyu(vol, sed, alternative='two-sided')
    med_diff = np.median(vol) - np.median(sed)
    print(f'MWU Volcanic vs Sedimentary {pc}: U={U:.0f}, p={p:.3e}, '
          f'median diff={med_diff:+.3f}')

print()
# Also test raw Hg concentration by broad class
hg_ok = lithology_ok.dropna(subset=['usgs_hg'])
groups_hg = [hg_ok.loc[hg_ok['broad_class'] == c, 'usgs_hg'].values for c in classes]
H_hg, p_hg = kruskal(*groups_hg)
print(f'Kruskal-Wallis log(Hg+1) ~ broad_class:  H={H_hg:.2f}, p={p_hg:.3e}')
for c in sorted(classes):
    grp = hg_ok.loc[hg_ok['broad_class'] == c, 'usgs_hg']
    print(f'  {c:<25} median Hg={grp.median():.4f} ppm, n={len(grp)}')


In [5]:
# ── Figure: PC1 / PC2 / Hg by lithology class ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(FIGW['full'], ROW_H),
                         gridspec_kw={'wspace': 0.35})

CLASS_ORDER = ['Volcanic', 'Sedimentary']
CLASS_COLORS = [PALETTE[3], PALETTE[0]]   # orange-ish, blue

lith_ok = merged.dropna(subset=['broad_class'])
lith_ok = lith_ok[lith_ok['broad_class'].isin(CLASS_ORDER)]

# PC1 boxplot
ax = axes[0]; grid_h(ax)
parts = ax.violinplot(
    [lith_ok.loc[lith_ok['broad_class'] == c, 'PC1'].values for c in CLASS_ORDER],
    positions=range(len(CLASS_ORDER)), showmedians=True, widths=0.7)
for i, (body, col) in enumerate(zip(parts['bodies'], CLASS_COLORS)):
    body.set_facecolor(col); body.set_alpha(0.6)
parts['cmedians'].set_color('k'); parts['cmedians'].set_linewidth(1.5)
for key in ('cbars', 'cmins', 'cmaxes'):
    parts[key].set_color('k'); parts[key].set_linewidth(0.8)
ax.set_xticks(range(len(CLASS_ORDER))); ax.set_xticklabels(CLASS_ORDER)
ax.set_ylabel(f'PC1 score ({pct[0]:.1f}% var) — lithogenic axis')
ax.set_title('PC1 by lithology', fontsize=10)
ax.axhline(0, color='gray', lw=0.8, ls='--')

# PC2 boxplot
ax = axes[1]; grid_h(ax)
parts = ax.violinplot(
    [lith_ok.loc[lith_ok['broad_class'] == c, 'PC2'].values for c in CLASS_ORDER],
    positions=range(len(CLASS_ORDER)), showmedians=True, widths=0.7)
for i, (body, col) in enumerate(zip(parts['bodies'], CLASS_COLORS)):
    body.set_facecolor(col); body.set_alpha(0.6)
parts['cmedians'].set_color('k'); parts['cmedians'].set_linewidth(1.5)
for key in ('cbars', 'cmins', 'cmaxes'):
    parts[key].set_color('k'); parts[key].set_linewidth(0.8)
ax.set_xticks(range(len(CLASS_ORDER))); ax.set_xticklabels(CLASS_ORDER)
ax.set_ylabel(f'PC2 score ({pct[1]:.1f}% var) — hydrothermal axis')
ax.set_title('PC2 by lithology\n(Hg/Se loading = +0.365)', fontsize=10)
ax.axhline(0, color='gray', lw=0.8, ls='--')

# Raw Hg by lithology
ax = axes[2]; grid_h(ax)
hg_ok2 = lith_ok.dropna(subset=['usgs_hg'])
parts = ax.violinplot(
    [np.log1p(hg_ok2.loc[hg_ok2['broad_class'] == c, 'usgs_hg'].values)
     for c in CLASS_ORDER],
    positions=range(len(CLASS_ORDER)), showmedians=True, widths=0.7)
for i, (body, col) in enumerate(zip(parts['bodies'], CLASS_COLORS)):
    body.set_facecolor(col); body.set_alpha(0.6)
parts['cmedians'].set_color('k'); parts['cmedians'].set_linewidth(1.5)
for key in ('cbars', 'cmins', 'cmaxes'):
    parts[key].set_color('k'); parts[key].set_linewidth(0.8)
ax.set_xticks(range(len(CLASS_ORDER))); ax.set_xticklabels(CLASS_ORDER)
ax.set_ylabel('log(USGS Hg ppm + 1)')
ax.set_title('Soil Hg concentration\nby lithology', fontsize=10)

fig.suptitle('GLiM lithology confirms PC2 = hydrothermal/volcanic Hg axis', y=1.02)
save(fig, FIGS / 'fig_nb36_lithology_pcs')
print('Saved fig_nb36_lithology_pcs.pdf')


In [1]:
# ── Figure: PCA biplot coloured by lithology ───────────────────────────────
from matplotlib.lines import Line2D

fig, axes = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H),
                         gridspec_kw={'wspace': 0.35})

# Panel A: sample scores coloured by broad lithology
ax = axes[0]; grid_h(ax)

col_map = {'Volcanic': PALETTE[3], 'Sedimentary': PALETTE[0]}
for cls, grp in merged.dropna(subset=['broad_class']).groupby('broad_class'):
    if cls not in col_map: continue
    ax.scatter(grp['PC1'], grp['PC2'],
               color=col_map[cls], alpha=0.35, s=8, edgecolor='none',
               label=f'{cls} (n={len(grp):,})')

# Overlay Hg and merA-tracked loading vectors (scaled for visibility)
MWAS_HG_MERA_TRACKED = ['nd','sb','ce','nb','ba','se','yb','y','as','sn',
                         'la','tl','sr','u','pb','zr','mo','bi','th','w','v','pd']
mera_avail = [m for m in MWAS_HG_MERA_TRACKED if m in loadings.index]
scale = 10.0
for metal in ['hg'] + mera_avail[:6]:
    if metal not in loadings.index: continue
    col = 'crimson' if metal == 'hg' else 'navy'
    ax.annotate('', xy=(loadings.loc[metal, 'PC1'] * scale,
                         loadings.loc[metal, 'PC2'] * scale),
                 xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color=col, lw=1.0))
    ax.text(loadings.loc[metal, 'PC1'] * scale * 1.08,
            loadings.loc[metal, 'PC2'] * scale * 1.08,
            metal.upper(), fontsize=6.5, color=col, ha='center')

legend_els = [mpatches.Patch(facecolor=col_map['Volcanic'],    label='Volcanic'),
              mpatches.Patch(facecolor=col_map['Sedimentary'], label='Sedimentary'),
              Line2D([0],[0], color='crimson', lw=1.2, label='Hg loading'),
              Line2D([0],[0], color='navy',    lw=1.2, label='merA-tracked metals')]
ax.legend(handles=legend_els, fontsize=7, loc='lower right', framealpha=0.8)
ax.set_xlabel(f'PC1 ({pct[0]:.1f}%)')
ax.set_ylabel(f'PC2 ({pct[1]:.1f}%)')
ax.set_title('PCA scores by lithology\n(vectors = PC loadings × 10)', fontsize=10)

# Panel B: Metal loadings bar chart annotated with geological meaning
ax = axes[1]; grid_h(ax)

# Top-loading metals on PC2 (absolute value), coloured by geologic process
top_pc2 = loadings['PC2'].abs().nlargest(12).index
load_df = loadings.loc[top_pc2].sort_values('PC2', ascending=True)

bar_colors = []
for m in load_df.index:
    if m in ['hg', 'se', 'cs', 'bi', 'sb']:  # hydrothermal/volcanic tracers
        bar_colors.append(PALETTE[3])
    elif m in ['ba', 'sr', 'zr', 'nb', 'la', 'ce', 'th', 'u']:  # crustal/lithogenic
        bar_colors.append(PALETTE[0])
    else:
        bar_colors.append('#aaaaaa')

ax.barh(range(len(load_df)), load_df['PC2'].values,
        color=bar_colors, edgecolor='k', linewidth=0.5)
ax.set_yticks(range(len(load_df)))
ax.set_yticklabels([m.upper() for m in load_df.index], fontsize=7)
ax.axvline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('PC2 loading')
ax.set_title('Top PC2 loadings\nby geological process', fontsize=10)
legend_els = [mpatches.Patch(facecolor=PALETTE[3], label='Hydrothermal/volcanic'),
              mpatches.Patch(facecolor=PALETTE[0], label='Lithogenic/crustal'),
              mpatches.Patch(facecolor='#aaaaaa',  label='Mixed')]
ax.legend(handles=legend_els, fontsize=7, loc='lower right', framealpha=0.8)

fig.suptitle('PC2 = hydrothermal/volcanic geochemistry; PC1 = crustal lithogenic', y=1.02)
save(fig, FIGS / 'fig_nb36_pca_biplot_lithology')
print('Saved fig_nb36_pca_biplot_lithology.pdf')


In [7]:
# ── Fine-grained: PC2 across all 6 GLiM classes present in dataset ─────────
from scipy.stats import kruskal

fine_ok = merged.dropna(subset=['lithology_class', 'PC2'])
# Keep classes with n >= 10
class_counts = fine_ok['lithology_class'].value_counts()
keep_classes = class_counts[class_counts >= 10].index.tolist()
fine_ok = fine_ok[fine_ok['lithology_class'].isin(keep_classes)].copy()
print(f'Fine-grained analysis: {len(keep_classes)} classes, n={len(fine_ok)}')

groups = [fine_ok.loc[fine_ok['lithology_class'] == c, 'PC2'].values
          for c in keep_classes]
H, p = kruskal(*groups)
print(f'Kruskal-Wallis PC2 ~ lithology_class (fine): H={H:.2f}, p={p:.3e}')

for c in sorted(keep_classes, key=lambda c: fine_ok.loc[fine_ok['lithology_class']==c, 'PC2'].median(), reverse=True):
    grp = fine_ok.loc[fine_ok['lithology_class'] == c, 'PC2']
    print(f'  {c:<40} median PC2={grp.median():+.3f}, n={len(grp)}')


In [8]:
# ── Summary ─────────────────────────────────────────────────────────────────
from scipy.stats import mannwhitneyu

lith_ok = merged.dropna(subset=['broad_class'])
lith_ok = lith_ok[lith_ok['broad_class'].isin(['Volcanic', 'Sedimentary'])]

pc2_vol = lith_ok.loc[lith_ok['broad_class'] == 'Volcanic',    'PC2'].values
pc2_sed = lith_ok.loc[lith_ok['broad_class'] == 'Sedimentary', 'PC2'].values
pc1_vol = lith_ok.loc[lith_ok['broad_class'] == 'Volcanic',    'PC1'].values
pc1_sed = lith_ok.loc[lith_ok['broad_class'] == 'Sedimentary', 'PC1'].values

_, p_pc1 = mannwhitneyu(pc1_vol, pc1_sed, alternative='two-sided')
_, p_pc2 = mannwhitneyu(pc2_vol, pc2_sed, alternative='two-sided')

hg_ok = lith_ok.dropna(subset=['usgs_hg'])
hg_vol = hg_ok.loc[hg_ok['broad_class'] == 'Volcanic',    'usgs_hg'].values
hg_sed = hg_ok.loc[hg_ok['broad_class'] == 'Sedimentary', 'usgs_hg'].values
_, p_hg = mannwhitneyu(hg_vol, hg_sed, alternative='two-sided')

print('=' * 65)
print('NB36 SUMMARY: GLiM lithology validates PCA axis interpretation')
print('=' * 65)
print()
print('PC2 (hydrothermal/volcanic axis):')
print(f'  Volcanic median  = {np.median(pc2_vol):+.3f}')
print(f'  Sedimentary med  = {np.median(pc2_sed):+.3f}')
print(f'  MWU p            = {p_pc2:.3e}  => {"SIGNIFICANT" if p_pc2 < 0.05 else "NS"}')
print()
print('PC1 (lithogenic crustal axis):')
print(f'  Volcanic median  = {np.median(pc1_vol):+.3f}')
print(f'  Sedimentary med  = {np.median(pc1_sed):+.3f}')
print(f'  MWU p            = {p_pc1:.3e}  => {"SIGNIFICANT — unexpected" if p_pc1 < 0.001 else "weaker or NS"}')
print()
print('Raw USGS Hg (ppm):')
print(f'  Volcanic median  = {np.median(hg_vol):.5f} ppm')
print(f'  Sedimentary med  = {np.median(hg_sed):.5f} ppm')
print(f'  MWU p            = {p_hg:.3e}  => {"SIGNIFICANT" if p_hg < 0.05 else "NS"}')
print()
print('Interpretation:')
if p_pc2 < 0.05:
    print('  PC2 IS significantly elevated in volcanic vs sedimentary sites.')
    print('  This confirms PC2 = hydrothermal/volcanic Hg enrichment axis.')
else:
    print('  PC2 not significantly elevated — check fine-grained classes.')
if p_pc1 >= 0.001:
    print('  PC1 shows weaker/no lithology dependence — consistent with')
    print('  crustal geochemistry spanning multiple rock types.')
